# Embedding Model Comparison -- Thesis Figures (Kap. 6.1)

Generates publication-quality figures from the 78-question full LeoWiki corpus evaluation.

**Results path:** `../results/results_of_full_wiki_corpus_78q/`

**Output:** Figures saved to `../figures/` (300 DPI PNG, LaTeX-ready).

**Colors:** Thesis theme `00_thesis_default` (HTL Leonding Corporate Design: primary #2E4F8F, secondary #72ADCB, accent #F28D2C).

In [ ]:
import json
import math
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Resolve paths: run from evaluation/notebooks, evaluation/, or repo root
cwd = Path.cwd()
cand1 = cwd / "results" / "results_of_full_wiki_corpus_78q"
cand2 = cwd.parent / "results" / "results_of_full_wiki_corpus_78q"
cand3 = cwd / "evaluation" / "results" / "results_of_full_wiki_corpus_78q"
if cand1.exists():
    RESULTS_DIR, FIGURES_DIR = cand1, cwd / "figures"
elif cand2.exists():
    RESULTS_DIR, FIGURES_DIR = cand2, cwd.parent / "figures"
else:
    RESULTS_DIR, FIGURES_DIR = cand3, cwd / "evaluation" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DPI = 300
plt.rcParams.update({'figure.dpi': DPI, 'savefig.dpi': DPI, 'font.family': 'serif'})
sns.set_theme(style='whitegrid', font='serif')
# Thesis theme (00_thesis_default: HTL Leonding)
THEME = {'primary': '#2E4F8F', 'secondary': '#72ADCB', 'accent': '#F28D2C', 'blue_medium': '#4671A3', 'neutral': '#8FA3A3', 'error': '#B33130', 'blue_pale': '#A5C9DC'}
THEME_COLORS = [THEME['primary'], THEME['secondary'], THEME['accent'], THEME['blue_medium'], THEME['neutral']]
sns.set_palette(THEME_COLORS)

def load_all_results():
    """Load all model_*.json files and return list of (short_name, data)."""
    rows = []
    for p in sorted(RESULTS_DIR.glob("model_*.json")):
        with open(p, encoding="utf-8") as f:
            data = json.load(f)
        name = data["experiment"]["name"]
        short = name.split("(")[0].strip() if "(" in name else name
        if "Octen" in short:
            short = "Octen-4B"
        elif "text-embedding" in short:
            short = "OpenAI 3-large"
        elif "PIXIE" in short:
            short = "PIXIE-Rune"
        elif "snowflake" in short:
            short = "Snowflake Arctic"
        elif "bge-m3" in short:
            short = "bge-m3-unsup"
        rows.append((short, data))
    return rows

results = load_all_results()
print(f"Loaded {len(results)} models")
for short, _data in results:
    print(f"  - {short}")

## Fig 6.1 -- Grouped bar chart (5 models x 4 metrics)

In [30]:
metrics = ["MRR", "NDCG@10", "Recall@10", "Hit Rate"]
keys = ["mrr", "ndcg_at_10", "recall_at_10", "hit_rate"]
x = np.arange(len(metrics))
n_models = len(results)
width = 0.8 / max(n_models, 1)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (short, d) in enumerate(results):
    agg = d["aggregate_metrics"]
    vals = []
    for k in keys:
        v = agg.get(k)
        if isinstance(v, dict):
            vals.append(v.get("mean", 0))
        else:
            vals.append(float(v) if v is not None else 0)
    offset = (i - n_models / 2 + 0.5) * width
    ax.bar(x + offset, vals, width, label=short)

ax.set_ylabel("Score")
ax.set_title("Embedding Model Comparison (78 Q&A, full corpus)")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_6_1_bar_comparison.png", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved fig_6_1_bar_comparison.png")

Saved fig_6_1_bar_comparison.png


## Fig 6.2 -- Radar chart (5 models, 6 metrics)

In [31]:
radar_metrics = ["MRR", "NDCG@10", "P@5", "R@10", "MAP", "Hit Rate"]
radar_keys = ["mrr", "ndcg_at_10", "precision_at_5", "recall_at_10", "map", "hit_rate"]
n_radar = len(radar_metrics)
angles = [m / float(n_radar) * 2 * math.pi for m in range(n_radar)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
for short, d in results:
    agg = d["aggregate_metrics"]
    vals = []
    for k in radar_keys:
        v = agg.get(k)
        if isinstance(v, dict):
            vals.append(v.get("mean", 0))
        else:
            vals.append(float(v) if v is not None else 0)
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2, label=short)
    ax.fill(angles, vals, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics)
ax.set_ylim(0, 1.05)
ax.set_title("Retrieval metrics by model", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_6_2_radar.png", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved fig_6_2_radar.png")

Saved fig_6_2_radar.png


## Fig 6.3 -- Performance by difficulty (all 5 models)

In [32]:
diffs = ["easy", "medium", "hard"]
x = np.arange(len(diffs))
n_models = len(results)
width = 0.75 / max(n_models, 1)

fig, ax = plt.subplots(figsize=(8, 5))
for i, (short, d) in enumerate(results):
    by_diff = d.get("by_difficulty", {})
    vals = [by_diff.get(diff, {}).get("mrr", 0) for diff in diffs]
    offset = (i - (n_models - 1) / 2) * width
    ax.bar(x + offset, vals, width, label=short)

ax.set_ylabel("MRR")
ax.set_title("MRR by question difficulty (all 5 models)")
ax.set_xticks(x)
ax.set_xticklabels(diffs)
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_6_3_by_difficulty.png", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved fig_6_3_by_difficulty.png")

Saved fig_6_3_by_difficulty.png


## Fig 6.4 -- Embedding speed vs. quality (scatter)

In [ ]:
chunks = results[0][1]["performance"]["corpus_chunks"]
sizes = []  # approx param size for bubble: 0.5=568M, 4=4B, 10=API
for short, d in results:
    if "Octen" in short or "4B" in d["experiment"].get("model", ""):
        sizes.append(4.0)
    elif "OpenAI" in short:
        sizes.append(10.0)
    else:
        sizes.append(0.568)

mrr_vals = [d["aggregate_metrics"].get("mrr", {}).get("mean", 0) or 0 for _, d in results]
speed_vals = []
for _, d in results:
    t = d["performance"].get("embedding_time_seconds") or 1
    speed_vals.append(chunks / t)

fig, ax = plt.subplots(figsize=(8, 6))
names = [r[0] for r in results]
for i, name in enumerate(names):
    ax.scatter(mrr_vals[i], speed_vals[i], s=400 + sizes[i] * 120, alpha=0.8, label=name, edgecolors="black", linewidths=0.5)
    ax.annotate(name, (mrr_vals[i], speed_vals[i]), xytext=(6, 6), textcoords="offset points", fontsize=9)
ax.set_xlabel("MRR")
ax.set_ylabel("Chunks per second")
ax.set_title("Embedding speed vs. retrieval quality")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=True)
x_min, x_max = min(mrr_vals), max(mrr_vals)
y_min, y_max = min(speed_vals), max(speed_vals)
ax.set_xlim(x_min - 0.02 * (x_max - x_min or 1), x_max + 0.02 * (x_max - x_min or 1) + 0.01)
ax.set_ylim(y_min - 0.05 * (y_max - y_min or 1), y_max + 0.15 * (y_max - y_min or 1))
fig.tight_layout(rect=[0, 0, 0.82, 1])
fig.savefig(FIGURES_DIR / "fig_6_4_speed_vs_quality.png", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved fig_6_4_speed_vs_quality.png")

## Fig 6.5 -- Per-query MRR distribution (box plot)

*Note: RR is bounded in [0, 1]; many queries score 1.0 (correct doc at rank 1), so upper quartiles and whiskers naturally reach the ceiling. Boxes that collapse to a line at 1.0 indicate a high share of perfect scores.*

In [34]:
rr_by_model = []
labels = []
for short, d in results:
    per = d.get("per_query", [])
    rr = [q.get("rr", 0) for q in per if q.get("rr") is not None]
    rr_by_model.append(rr)
    labels.append(short)

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(rr_by_model, tick_labels=labels, patch_artist=True, showmeans=True,
                 meanprops={"marker": "D", "markerfacecolor": "white", "markersize": 6})
for patch, col in zip(bp["boxes"], THEME_COLORS, strict=False):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
ax.set_ylabel("Reciprocal Rank (per query)")
ax.set_title("Per-query MRR distribution by model")
ax.set_ylim(-0.05, 1.18)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
fig.subplots_adjust(top=0.92)
fig.savefig(FIGURES_DIR / "fig_6_5_mrr_boxplot.png", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved fig_6_5_mrr_boxplot.png")

Saved fig_6_5_mrr_boxplot.png


## Fig 6.5b–6.5e -- Per-query RR: academic-standard alternatives

Violin (full distribution shape), ECDF (cumulative fraction), strip plot (individual points), and binned bar (RR=0 / 0<RR<1 / RR=1) avoid the bounded-metric illusion of box plots and are common in IR/thesis literature.

In [35]:
# Long-format dataframe: one row per (model, query) with RR value
import pandas as pd

rr_rows = []
for short, d in results:
    for q in d.get("per_query", []):
        rr = q.get("rr")
        if rr is not None:
            rr_rows.append({"model": short, "RR": float(rr)})
df_rr = pd.DataFrame(rr_rows)
if df_rr.empty:
    print("[WARN] No per-query RR data found.")
else:
    print(f"Per-query RR: {len(df_rr)} points across {df_rr['model'].nunique()} models.")

Per-query RR: 390 points across 5 models.


In [36]:
# Fig 6.5b -- Violin plot (full distribution shape; pile-up at 0 and 1 visible)
if not df_rr.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    order = [r[0] for r in results]
    sns.violinplot(data=df_rr, x="model", y="RR", order=order, palette=THEME_COLORS, ax=ax)
    ax.set_ylabel("Reciprocal Rank (per query)")
    ax.set_title("Per-query RR distribution by model (violin)")
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    plt.xticks(rotation=15, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_6_5b_violin.png", bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    print("Saved fig_6_5b_violin.png")

Saved fig_6_5b_violin.png


C:\Users\janri\AppData\Local\Temp\ipykernel_19724\2165181749.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df_rr, x="model", y="RR", order=order, palette=THEME_COLORS, ax=ax)


In [37]:
# Fig 6.5c -- ECDF (Empirical CDF: fraction of queries with RR <= x)
if not df_rr.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    for (short, _), col in zip(results, THEME_COLORS, strict=False):
        vals = np.sort(df_rr.loc[df_rr["model"] == short, "RR"].values)
        n = len(vals)
        if n > 0:
            y = np.arange(1, n + 1, dtype=float) / n
            ax.step(np.r_[0, vals, 1.01], np.r_[0, y, 1], where="post", label=short, color=col)
    ax.set_xlabel("Reciprocal Rank")
    ax.set_ylabel("P(RR <= x)")
    ax.set_title("Empirical CDF of per-query RR by model")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_6_5c_ecdf.png", bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    print("Saved fig_6_5c_ecdf.png")

Saved fig_6_5c_ecdf.png


In [38]:
# Fig 6.5d -- Strip plot with jitter (individual points visible)
if not df_rr.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    order = [r[0] for r in results]
    sns.stripplot(data=df_rr, x="model", y="RR", order=order, palette=THEME_COLORS, jitter=0.2, size=4, alpha=0.7, ax=ax)
    ax.set_ylabel("Reciprocal Rank (per query)")
    ax.set_title("Per-query RR by model (strip plot)")
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    plt.xticks(rotation=15, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_6_5d_strip.png", bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    print("Saved fig_6_5d_strip.png")

Saved fig_6_5d_strip.png


C:\Users\janri\AppData\Local\Temp\ipykernel_19724\329917716.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(data=df_rr, x="model", y="RR", order=order, palette=THEME_COLORS, jitter=0.2, size=4, alpha=0.7, ax=ax)


In [39]:
# Fig 6.5e -- Binned bar (RR=0, 0<RR<1, RR=1) per model
if not df_rr.empty:
    bins = ["RR = 0", "0 < RR < 1", "RR = 1"]
    order = [r[0] for r in results]
    counts = {short: [0, 0, 0] for short in order}
    for _, row in df_rr.iterrows():
        rr = row["RR"]
        m = row["model"]
        if rr <= 0:
            counts[m][0] += 1
        elif rr >= 1:
            counts[m][2] += 1
        else:
            counts[m][1] += 1
    x = np.arange(len(order))
    w = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (label, col) in enumerate(zip(bins, [THEME["error"], THEME["neutral"], THEME["primary"]], strict=False)):
        vals = [counts[m][i] for m in order]
        ax.bar(x + (i - 1) * w, vals, w, label=label, color=col)
    ax.set_ylabel("Number of queries")
    ax.set_title("Per-query RR bins by model")
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=15, ha="right")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_6_5e_binned.png", bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    print("Saved fig_6_5e_binned.png")

Saved fig_6_5e_binned.png


## LLM-as-Judge (RAGAS-style) metrics

If the evaluation was run with LLM-as-Judge (without `--skip ragas`), result files may contain `ragas_metrics` or `llm_judge` (faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness). Below: load if present and plot bar comparison and radar; otherwise show how to generate the data.

In [40]:
# Load LLM-as-Judge metrics if present (ragas_metrics / llm_judge in model JSONs or combined_results.json)
JUDGE_KEYS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness", "ragas_score"]
judge_results = []
for short, d in results:
    j = d.get("ragas_metrics") or d.get("llm_judge") or {}
    if j and any(k in j for k in JUDGE_KEYS):
        judge_results.append((short, {k: (float(v) if isinstance(v, (int, float)) else None) for k, v in j.items() if k in JUDGE_KEYS}))
if not judge_results and (RESULTS_DIR / "combined_results.json").exists():
    with open(RESULTS_DIR / "combined_results.json", encoding="utf-8") as f:
        comb = json.load(f)
    j = comb.get("ragas_metrics") or comb.get("llm_judge")
    if j:
        judge_results = [(comb.get("experiment", {}).get("name", "run"), j)]

if judge_results:
    # Bar: models x judge metrics
    judge_metrics = [k for k in JUDGE_KEYS if any(r[1].get(k) is not None for r in judge_results)]
    if judge_metrics:
        x = np.arange(len(judge_metrics))
        n_m = len(judge_results)
        width = 0.8 / max(n_m, 1)
        fig, ax = plt.subplots(figsize=(max(8, len(judge_metrics) * 1.5), 5))
        for i, (short, m) in enumerate(judge_results):
            vals = [m.get(k) or 0 for k in judge_metrics]
            ax.bar(x + (i - (n_m - 1) / 2) * width, vals, width, label=short, color=THEME_COLORS[i % len(THEME_COLORS)])
        ax.set_ylabel("Score")
        ax.set_title("LLM-as-Judge metrics by model")
        ax.set_xticks(x)
        ax.set_xticklabels([k.replace("_", " ").title() for k in judge_metrics], rotation=25, ha="right")
        ax.set_ylim(0, 1.05)
        ax.legend(loc="lower right")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "fig_llm_judge_bar.png", bbox_inches="tight", dpi=DPI)
        plt.close(fig)
        print("Saved fig_llm_judge_bar.png")
        # Radar (if at least 3 metrics)
        if len(judge_metrics) >= 3:
            n_radar = len(judge_metrics)
            angles = [m / float(n_radar) * 2 * math.pi for m in range(n_radar)]
            angles += angles[:1]
            fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
            for i, (short, m) in enumerate(judge_results):
                vals = [m.get(k) or 0 for k in judge_metrics]
                vals += vals[:1]
                ax.plot(angles, vals, "o-", linewidth=2, label=short, color=THEME_COLORS[i % len(THEME_COLORS)])
                ax.fill(angles, vals, alpha=0.1)
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels([k.replace("_", " ").title() for k in judge_metrics])
            ax.set_ylim(0, 1.05)
            ax.set_title("LLM-as-Judge metrics (radar)", fontweight="bold", pad=20)
            ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
            fig.tight_layout()
            fig.savefig(FIGURES_DIR / "fig_llm_judge_radar.png", bbox_inches="tight", dpi=DPI)
            plt.close(fig)
            print("Saved fig_llm_judge_radar.png")
else:
    print("[INFO] No LLM-as-Judge data found in results. To generate: run eval pipeline without --skip ragas, or add LLM-judge step to model comparison and re-run.")

[INFO] No LLM-as-Judge data found in results. To generate: run eval pipeline without --skip ragas, or add LLM-judge step to model comparison and re-run.


## Fig 6.6 -- Miss analysis (model x question hit/miss heatmap)

In [41]:
query_ids = []
if results:
    query_ids = [q.get("id", str(i)) for i, q in enumerate(results[0][1].get("per_query", []))]
n_q = len(query_ids)
n_m = len(results)
hit_matrix = np.zeros((n_m, n_q))
for i, (_, d) in enumerate(results):
    for j, q in enumerate(d.get("per_query", [])):
        hit_matrix[i, j] = 1.0 if q.get("hit_in_top_k") else 0.0

from matplotlib.colors import ListedColormap

thesis_cmap = ListedColormap([THEME['error'], THEME['primary']])
fig, ax = plt.subplots(figsize=(max(12, n_q * 0.15), max(5, n_m * 1.2)))
sns.heatmap(hit_matrix, xticklabels=query_ids, yticklabels=[r[0] for r in results],
            cmap=thesis_cmap, vmin=0, vmax=1, cbar_kws={"label": "Hit (1) / Miss (0)"}, ax=ax)
ax.set_title("Hit/Miss by model and question")
ax.set_xlabel("Question ID")
plt.xticks(rotation=90, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_6_6_miss_heatmap.png", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved fig_6_6_miss_heatmap.png")

Saved fig_6_6_miss_heatmap.png


---
All figures saved to `evaluation/figures/`. Use in LaTeX with `\includegraphics{fig_6_1_bar_comparison}` etc.